In [1]:
# at the very top of volatility_analysis.py
import sys, os
sys.path.insert(0, os.path.abspath("c:/Users/marku/OneDrive/Dokumente/WUTIS/SS25 Volatility Prediction w RNN/wutis-rnn-vol"))

import pandas as pd
import json
from IPython.display import display
from pathlib import Path
import numpy as np
import optuna
from optuna.visualization import plot_optimization_history, plot_param_importances, plot_contour

import matplotlib.pyplot as plt
from trading_window.optim_slots import (
    DataSplitter,
    MarketEnvironment,
    RSIStrategy,
    QLearningAgent,
    train_q_learning_and_extract_dates
    
)
from trading_window.finding_slots import TradingWindowOptimizer, DEFAULT_ALPHA, DEFAULT_BETA


PROJECT_ROOT = Path.cwd()  
# make sure the folder exists
optuna_dir = PROJECT_ROOT / "optuna_studies"
optuna_dir.mkdir(exist_ok=True)

db_path = optuna_dir / "window_search.sqlite3"
storage_name = f"sqlite:///{db_path}"

In [2]:
study_name = "APL_trading_window_optimization_80_20_oter"
file_path = 'C:/Users/marku/OneDrive/Dokumente/WUTIS/SS25 Volatility Prediction w RNN/data/train_data.csv'
# 
# Read the CSV file
df = pd.read_csv(file_path, parse_dates=['timestamp'])

# …then, assuming you’ve already loaded your full-minute DataFrame `df`:
df['date'] = pd.to_datetime(df['timestamp']).dt.date

# from hyperparameter_optimizer import TradingWindowOptimizer
optimizer = TradingWindowOptimizer(
    df,
    MarketEnvironment,
    RSIStrategy,
    DataSplitter.split_by_date,
    alpha = 0.8, # Volatility
    beta = 0.2, # Volume
    gamma = 0.1, # Profit
    min_window_size=15,  # Minimum window size
    max_window_size=720  # Maximum window size 
)
study = optimizer.optimize(n_trials=2, 
                           study_name = study_name, 
                           storage_name = storage_name)

print(study.best_params)


[I 2025-06-10 22:17:38,517] Using an existing study with name 'APL_trading_window_optimization_80_20_oter' instead of creating a new one.
[I 2025-06-10 22:17:59,559] Trial 61 finished with value: 0.18040241517007968 and parameters: {'n_days': 116, 'window_size': 168}. Best is trial 42 with value: 0.7999999999999999.
[I 2025-06-10 22:18:14,229] Trial 62 finished with value: 0.22004258947351088 and parameters: {'n_days': 54, 'window_size': 47}. Best is trial 42 with value: 0.7999999999999999.


{'n_days': 107, 'window_size': 24}


In [3]:
# optuna.delete_study(study_name = study_name, storage  = storage_name)	

In [4]:
study = optuna.load_study(
    study_name=study_name, storage=storage_name
    )


# after running
# study = optimizer.optimize(n_trials=100)

# 1) How the best objective value evolved over trials:
fig1 = plot_optimization_history(study)
fig1.show()

# 2) Which hyperparameters ended up most important:
fig2 = plot_param_importances(study)
fig2.show()


# days = DataSplitter.split_by_date(df)
# env = MarketEnvironment(days[0], RSIStrategy())
# agent=QLearningAgent(n_states=100, actions=['hold','buy','sell'], alpha=0.1, gamma=0.99, epsilon=1.0)
# rets, errors, eps = train_q_learning(env, agent, episodes=500)
# plt.plot(rets); plt.title('Episode Returns'); plt.show()
# plt.plot(errors); plt.title('TD Errors'); plt.show()
# plt.plot(eps); plt.title('Epsilon Decay'); plt.show()


In [5]:
best = study.best_params
print(best)

plot_contour(study, params=["n_days", "window_size"])


{'n_days': 107, 'window_size': 24}


In [10]:
one_day = DataSplitter.split_by_date(df)[0]
best_n = 107
best_w = 24

bars_per_day = len(one_day)
slots = list(range(0, bars_per_day, best_w))

# 4) Stage 2: Q‐learning to pick EXACTLY best_n days + slots
env   = MarketEnvironment(df, RSIStrategy())
n_states = len(env.calendar_days) * len(slots)
agent = QLearningAgent(n_states=n_states, alpha=0.1, gamma=0.99, epsilon=1.0)

chosen_calendar = train_q_learning_and_extract_dates(
    env         = env,
    agent       = agent,
    episodes    = 500,
    n_days      = best_n,
    window_size = best_w,
    slots       = slots
)

print("Learned (MM-DD → start_minute):")
for day, start in chosen_calendar:
    print(f"  {day} @ minute {start}")

Learned (MM-DD → start_minute):
  10-27 @ minute 168
  04-06 @ minute 0
  03-06 @ minute 168
  10-28 @ minute 0
  04-26 @ minute 144
  02-22 @ minute 120
  10-19 @ minute 192
  11-11 @ minute 168
  07-31 @ minute 96
  09-18 @ minute 168
  07-10 @ minute 168
  08-31 @ minute 120
  10-18 @ minute 264
  03-30 @ minute 96
  05-04 @ minute 624
  08-21 @ minute 120
  05-16 @ minute 48
  08-04 @ minute 168
  10-21 @ minute 216
  03-10 @ minute 144
  04-17 @ minute 192
  05-03 @ minute 648
  06-30 @ minute 144
  05-23 @ minute 96
  04-27 @ minute 96
  02-28 @ minute 120
  05-01 @ minute 0
  11-15 @ minute 600
  04-21 @ minute 96
  07-06 @ minute 216
  10-03 @ minute 672
  04-12 @ minute 96
  10-31 @ minute 168
  06-07 @ minute 216
  01-31 @ minute 240
  02-14 @ minute 144
  11-30 @ minute 240
  03-03 @ minute 240
  03-13 @ minute 120
  03-03 @ minute 120
  09-14 @ minute 144
  11-23 @ minute 528
  10-14 @ minute 408
  10-19 @ minute 288
  08-02 @ minute 264
  11-03 @ minute 312
  04-21 @ minut

In [ ]:
print(returns)

plt.plot(returns)
plt.show()


plt.plot(eps_hist)
plt.show()
